This document outlines the preprocessing steps I used to clean the raw text data.

In [2]:
import pickle
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

I loaded in my pkl dictionaries to identify all dockets that existed for both the amicus briefs and opinions.

In [10]:
with open("models/ab_dict.pkl", "rb") as f:
    ab_dict = pickle.load(f)

In [11]:
with open("models/op_dict.pikl", "rb") as f:
    op_dict = pickle.load(f)

Because the underlying motivation of this project is to identify linguistic similarities that could indicate avenues for future research operationalizing language as a measure of AB influence on SC rulings, I am only interested in comparing the language between briefs that have a corresponding ruling. To filter these out, I identified the docket numbers that were common between both corpuses, and moved forward with only the 78 (39 per corpus) documents with those docket numbers.

In [12]:
## COME BACK TO THESE WHEN PERFECTING THE PROJECT NEXT QUARTER
common_dockets = set(op_dict.keys()) & set(ab_dict.keys())

# Filter dictionaries to only keep common dockets
filtered_op_dict = {key: op_dict[key] for key in common_dockets}
filtered_ab_dict = {key: ab_dict[key] for key in common_dockets}

# Print to check the number of common dockets
print(f"Number of common dockets: {len(common_dockets)}")

Number of common dockets: 39


In [15]:
df = pd.read_csv('raw_text_filename.csv')
filtered_df = df[df["Docket Number"].isin(common_dockets)]
print(len(df))
# Display the results
len(filtered_df)

175


95

### Helper Functions Include: ###
- Functions to remove unwanted sections of the text (such as the table of contents, table of authorities, and appendices)
- Functions to remove special characters (supplemented by additional regular expressions in the actual text cleaning section)
- Function to remove stopwords
- Functions to identify part-of-speech tags and use them for lemmatization

In [9]:
# limits the text to everything after the table of contents/authorities
def clean_text(text):
    ''' 
    Attempts to remove all extraneous text from a document
    by targeting the place where the table of contents ends
    (defined as the last series of dots or hyphens)

    '''
    # Split text into lines
    lines = text.split("\n")
    
    # Find the last occurrence of a line with dots/hyphens followed by a number
    last_index = -1
    for i, line in enumerate(lines):
        if re.search(r"[\.\-]{5,}\s*\d+$", line):  # At least 5 dots/hyphens before a number
            last_index = i
    
    # If we found such a line, return everything after it
    if last_index != -1 and last_index + 1 < len(lines):
        return "\n".join(lines[last_index + 1:]).strip()
    
    return text.strip()  # If no match, return original text

def remove_conclusion_and_appendix(text):
    ''' 
    Uses regex to remove conclusion and appendix sections from
    amicus brief text (that weren't already handled by clean_text)
    input: text (a list of strings)
    output: a list of strings with conclusion and appendix sections
    removed
    '''
    # Look for "CONCLUSION" or "APPENDIX" appearing as a clear section heading
    pattern = r"\n\s*(CONCLUSION|CONCLUSIONS|APPENDIX)\s*\n"
    
    # Split text at the first strong match
    match = re.search(pattern, text, flags=re.IGNORECASE)
    
    if match:
        return text[:match.start()].strip()  # Keep everything before the match
    return text  # If no match, return original text

def remove_special_chars(text):
    ''' 
    Removes all non-alphanumeric special characters
    input: text (a list of strings)
    output: a list of strings with special characters removed
    '''
    return re.sub(r"[^a-zA-Z0-9\s]", "", text)

def remove_stopwords(text):
    ''' 
    Removes english and domain-specific stop words
    input: text (a list of strings)
    output: a list of strings with stopwords removed
    '''
    extra_words = ['table', 'contents', 'authorities', 'interest', 'interest', 
               'authority', 'content', 'amicus', 'amici', 'brief', 'conclusion', 
               'curia', 'curiae', 'curio', 'see', 'et al', 'slip', 'opinion', 'syllabus', 
               'october', 'term', 'catholic', 'united state', 'united states', 'ruling', 'ruling', 'et', 
               'al', 'id', 'court', 'prosecution', 'case', 'appeal', 'petitioner', 'petition', 
               'respondent', 'court', 'law', 'u', 'ed', 'act', 'right', 'state', 'one', 
               'clause', 'tion', 'fide', 'quo', 'burgh', 'per', 'fore', 'cite', 'appendix', 'supp', 'quote', 
               'sary', 'certoriari', ]
    stop_words = set(stopwords.words('english')).update(extra_words)
    stop_words = set(stopwords.words('english')) | set(extra_words)
    # tokenize the text into words
    words = word_tokenize(text)
    
    # filter out stopwords
    filtered_words = [word for word in words if word.lower() not in stop_words]
    
    # join words back into a sentence
    return ' '.join(filtered_words)

# The following functions are repurposed from the group project (I wrote them originally)
def get_pos_tag(word):
    ''' 
    Identifies the part of speech of a word
    (for utilization in lemmatization)
    Input: word (string)
    '''
    tag = pos_tag([word])[0][1]
    if tag.startswith('N'):
        return 'n'  # Noun
    elif tag.startswith('V'):
        return 'v'  # Verb
    elif tag.startswith('J'):
        return 'a'  # Adjective
    elif tag.startswith('R'):
        return 'r'  # Adverb
    else:
        return 'n'  # default to noun if no match
    
# global lemmatizer variable
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    '''
    Lemmatizes text with POS tags
    Input: text (string)
    Output: lemmatized version of string
    '''
    # tokenize the text into words
    words = word_tokenize(text)
    
    # lemmatize each word with POS tagging
    lemmatized_words = [
        lemmatizer.lemmatize(word, get_pos_tag(word)) for word in words
    ]
    
    return ' '.join(lemmatized_words)

### Cleaning the amicus brief text:

In [8]:
# read in briefs from raw text file
csv_file_path = '/Users/haileyhansen/MACSS/30100/macs-30100-hailhan/P3_catholic_kmeans/files/raw_text.csv'

df = pd.read_csv(csv_file_path, header=None)
df.columns = ['docket', 'text']  # Adjust as needed
df = df[df['docket'].isin(common_dockets)] # limiting the output to just those docket numbers that are shared
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/Users/haileyhansen/MACSS/30100/macs-30100-hailhan/P3_catholic_kmeans/files/raw_text.csv'

In [1]:
len(df)

NameError: name 'df' is not defined

You'll notice there are two chunks of regular expressions below. I found, with some trial and error, that applying the above functions, standardizing whitespace, removing numbers, and removing urls helped was best done before lemmatization, and the removal of less than three letter words, or words made up of the same letter repeated were best done after lemmatization. I think that lemmatization left some strange lemma behind, so it ended up working better to save those additional regex for after lemmatization.

In [ ]:
df["text"] = df["text"].apply(clean_text)
df['text'] = df['text'].replace(r'\.+\d+|\-+\d+|[\.\w]+\d+', '')
df['text'] = df['text'].apply(remove_conclusion_and_appendix)
df['text'] = df['text'].apply(remove_special_chars)
numbers = r'\d*'
df['text'] = df['text'].str.replace(numbers, '', regex=True)
standard_whitespace = r'\s+'
df['text'] = df['text'].str.replace(standard_whitespace, ' ', regex=True).str.lower()
urls = r'\b(?:https?)\s\S*'  # match http/https/www followed by non-space characters
df['text'] = df['text'].str.replace(urls, '', regex=True)
df['text'] = df['text'].apply(remove_stopwords)

df.head()

,docket,text
1,01-1118,catholics life sacra mento support peti tioner...
2,88-1503,noc nan cy beth cruzan parents coguardians les...
3,18-1323,john c wilke barbara h wilke abortion question...
7,89-1215,williams firing woman protect fetus reconcilia...
8,15-274,national medical organizations combined member...


In [13]:
df['text'] = df['text'].apply(lemmatize_text)
df.head()

,docket,text
1,01-1118,catholic life sacrum mento support peti tioner...
2,88-1503,noc nan cy beth cruzan parent coguardians lest...
3,18-1323,john c wilke barbara h wilke abortion question...
7,89-1215,williams fire woman protect fetus reconciliati...
8,15-274,national medical organization combine membersh...


In [14]:
non_vowel_pattern = r'\b[^aeiouAEIOU\s\d]+\b' # remove words w/o vowels
df['text'] = df['text'].str.replace(non_vowel_pattern, '', regex=True)
one_letter_words = r'\b\w\b'
df['text'] = df['text'].str.replace(one_letter_words, ' ', regex=True)
words_of_one_letter = r'\b(\w)\1*\b' # remove words consisting of the same letter repeating
df['text'] = df['text'].str.replace(words_of_one_letter, ' ', regex=True)
less_than_3_letters = r'\b\w{1,2}\b' # helped remove additional meaningless words
df['text'] = df['text'].str.replace(less_than_3_letters, '', regex=True)

In [15]:
df.head()

,docket,text
1,01-1118,catholic life sacrum mento support peti tioner...
2,88-1503,noc nan beth cruzan parent coguardians lester...
3,18-1323,john wilke barbara wilke abortion question a...
7,89-1215,williams fire woman protect fetus reconciliati...
8,15-274,national medical organization combine membersh...


In [12]:
# export cleaned text to csv
df.to_csv('cleaned_text.csv', header=True, index=False)

### Cleaning the ruling text:

In [13]:
op_file_path = '/Users/haileyhansen/MACSS/30100/macs-30100-hailhan/P3_catholic_kmeans/raw_op_text.csv'

op = pd.read_csv(op_file_path, header=None)
op.columns = ['docket', 'text']
op = op[op['docket'].isin(common_dockets)] 
# limiting the output to shared docket numbers
print(op.head())

     docket                                               text
0   18-1323  (Slip Opinion) OCTOBER TERM, 2019 1\nSyllabus\...
1    88-805  502 OCTOBER TERM, 1989\nSyllabus 497 U. S.\nOH...
8   81-2338  540 OCTOBER TERM, 1982\nSyllabus 461 U. S.\nRE...
15  88-1503  CRUZAN r. DIRECTOR, MISSOURI DEPT. OF HEALTH 2...
23   88-605  490 OCTOBER TERM, 1988\nSyllabus 492 U. S.\nWE...


In [14]:
op["text"] = op["text"].apply(clean_text)
op['text'] = op['text'].replace(r'\.+\d+|\-+\d+|[\.\w]+\d+', '')
op['text'] = op['text'].apply(remove_conclusion_and_appendix)
op['text'] = op['text'].apply(remove_special_chars)
numbers = r'\d*'
op['text'] = op['text'].str.replace(numbers, '', regex=True)
standard_whitespace = r'\s+'
op['text'] = op['text'].str.replace(standard_whitespace, ' ', regex=True).str.lower()
urls = r'\b(?:https?)\s\S*' 
op['text'] = op['text'].str.replace(urls, '', regex=True)
op['text'] = op['text'].apply(remove_stopwords)

op.head()

,docket,text
0,18-1323,note feasible headnote released done connectio...
1,88-805,ohio v akron center reproductive health united...
8,81-2338,regan secretary treasury v taxation representa...
15,88-1503,cruzan r director missouri dept health cruzan ...
23,88-605,webster attorney general missouri v reproducti...


In [15]:
op['text'] = op['text'].apply(lemmatize_text)
op.head()

,docket,text
0,18-1323,note feasible headnote release do connection t...
1,88-805,ohio v akron center reproductive health united...
8,81-2338,regan secretary treasury v taxation representa...
15,88-1503,cruzan r director missouri dept health cruzan ...
23,88-605,webster attorney general missouri v reproducti...


In [16]:
words_of_one_letter = r'\b(\w)\1*\b'
op['text'] = op['text'].str.replace(words_of_one_letter, ' ', regex=True)
non_vowel_pattern = r'\b[^aeiouAEIOU\s\d]+\b'
op['text'] = op['text'].str.replace(non_vowel_pattern, '', regex=True)
one_letter_words = r'\b\w\b'
op['text'] = op['text'].str.replace(one_letter_words, ' ', regex=True)
less_than_3_letters = r'\b\w{1,2}\b'
df['text'] = df['text'].str.replace(less_than_3_letters, '', regex=True)

In [17]:
# export the cleaned text to csv
op.to_csv('cleaned_op_text.csv', header=True, index=False)